# Give your agent the web with Tavily

One agent, built twice:

1. **Without web access.** It only knows what the model learned in training.
2. **With Tavily.** The same agent, plus two tools.

Stack: [LangChain Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview) for the agent, a model served by [Nebius Token Factory](https://tokenfactory.nebius.com/), and [Tavily](https://tavily.com) for the web.

## Setup

Copy `.env.example` to `.env` in the repo root and fill in `NEBIUS_API_KEY` and `TAVILY_API_KEY`.

## 1. An agent without web access

Pick a question whose answer changed after the model was trained.

In [7]:
import json
import os
import time

from deepagents import create_deep_agent
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain_core.messages import AIMessage, ToolMessage
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

missing = [k for k in ("NEBIUS_API_KEY", "TAVILY_API_KEY") if not os.getenv(k)]
assert not missing, (
    f"Missing {', '.join(missing)}. Copy .env.example to .env and add keys from "
    "https://tokenfactory.nebius.com/ and https://app.tavily.com/"
)

MODEL = "meta-models/muse-spark-open"

model = ChatOpenAI(
    model=MODEL,
    api_key=os.getenv("NEBIUS_API_KEY"),
    base_url="https://api.tokenfactory.nebius.com/v1/",
    extra_body={"chat_template_kwargs": {"allow_missing_thinking": True}},
)

SYSTEM_PROMPT = f"""Today is {time.strftime("%Y-%m-%d")}. You are a helpful assistant.
Answer the question directly. Cite a source URL for every fact you look up."""


def _truncate(value, max_chars=60):
    if isinstance(value, str):
        text = " ".join(value.split())
        return text if len(text) <= max_chars else text[: max_chars - 1] + "…"
    if isinstance(value, dict):
        return {
            k: _truncate(v, max_chars)
            for k, v in value.items()
            if v not in (None, [], "")
        }
    if isinstance(value, list):
        return [_truncate(v, max_chars) for v in value]
    return value


def _print_tool_result(name, content, max_line=160):
    if isinstance(content, str):
        try:
            content = json.loads(content)
        except json.JSONDecodeError:
            print(f"← {name} {_truncate(content, max_line)}", flush=True)
            return
    data = content.get("results", content) if isinstance(content, dict) else content
    line = json.dumps(_truncate(data), ensure_ascii=False)
    if len(line) > max_line:
        line = line[: max_line - 1] + "…"
    print(f"← {name} {line}", flush=True)


def ask(agent, question):
    """Stream the agent run, print each tool call and result live, and display the answer."""
    answer = ""
    stream = agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="updates",
    )
    for update in stream:
        for partial in (update or {}).values():
            if not isinstance(partial, dict):
                continue
            for message in partial.get("messages") or []:
                if isinstance(message, AIMessage):
                    for call in message.tool_calls or []:
                        print("→", call["name"], call["args"], flush=True)
                    if not message.tool_calls and message.text.strip():
                        answer = message.text
                elif isinstance(message, ToolMessage):
                    _print_tool_result(message.name or "tool", message.content)
    display(Markdown(answer))

In [8]:
QUESTION = "What are the three newest models on Nebius Token Factory, and when was each one added?"

agent = create_deep_agent(model=model, system_prompt=SYSTEM_PROMPT)

ask(agent, QUESTION)

→ ls {'path': '/'}
← ls No files found
→ glob {'pattern': '**/*'}
← glob No files found
→ grep {'pattern': 'Nebius Token Factory'}
← grep No matches found
→ read_file {'file_path': '/tmp/test.txt'}
← read_file Error: File '/tmp/test.txt' not found


 I don't have a live web-browsing tool available in this environment to check the current Nebius Token Factory catalog - my filesystem search tools (`ls`, `glob`, `grep`) returned no local files, and no web-search/fetch tool is exposed to me to retrieve `nebius.com` live, so I can't verify the *current* newest 3 models with a citable live lookup.

To get the authoritative, up-to-the-minute list with added dates, check directly:

1. **Nebius Token Factory explorer:** https://nebius.com/token-factory
2. **Docs / Models list:** https://docs.nebius.com/studio/inference/overview and https://studio.nebius.com/ (Inference Playground -> Models) - models show provider, context length, and date added/supported.

There you can sort/filter by "Date Added" - the 3 at the top are the newest and each model card lists `Added on: YYYY-MM-DD`.

If you paste a screenshot or copy the 3 model names/dates you see there, I can help summarize, compare pricing/context, or format the answer with proper source citations to https://nebius.com/token-factory .

My training cutoff is 2026-01-04, so any list I recalled would be outdated and I don't want to hallucinate dates without a live source URL to cite for each fact, per your citation requirement.


## 2. The same agent with Tavily

The only change is the `tools` argument. `TavilySearch` finds sources (with recency and domain filters) and `TavilyExtract` reads full pages as markdown.

In [9]:
from langchain_tavily import TavilyExtract, TavilySearch

web_agent = create_deep_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[TavilySearch(max_results=5, search_depth="fast"), TavilyExtract()],
)

ask(web_agent, QUESTION)

→ tavily_search {'query': 'Nebius Token Factory newest models added when', 'search_depth': 'advanced'}
→ tavily_search {'query': 'Nebius AI Studio Token Factory models list', 'search_depth': 'advanced'}
← tavily_search [{"url": "https://llmendpoint.com/providers/nebius-token-factory", "title": "Nebius Token Factory API: Pricing, Models & Alternatives | …", "score": 0.76100606…
← tavily_search [{"url": "https://www.linkedin.com/posts/nebius_token-factory-has-two…", "title": "Token Factory has two new models. | Nebius", "score": 0.7324005, "content": …
→ tavily_extract {'urls': ['https://x.com/NeBiuSTf', 'https://www.linkedin.com/posts/nebius_token-factory-has-two-new-models-activity-7520871520349536259-AxtV'], 'query': 'Nebius Token Factory new models added date', 'extract_depth': 'advanced'}
→ tavily_search {'query': '"Nebius Token Factory" DeepSeek-V4.1-Flash-1017 date added', 'search_depth': 'advanced'}
→ tavily_search {'query': '"Nebius Token Factory" MiniMax M3.1 Flash date added',

As of 2026-11-23, the three newest announced models on Nebius Token Factory are:

1. **Inkling 2 Small (from Thinking Machines)** - Added **Nov 12, 2026**
   - Announced as: "Inkling 2 Small is available on Nebius Token Factory from Day 0. Build agents that write code, use tools, inspect visual results and iterate." on Nov 12 [https://x.com/nebiustf?lang=en](https://x.com/nebiustf?lang=en)
   - Also mirrored: "Inkling-2-Small, small yet powerful, now available on Nebius Token Factory from Day 0" on Nov 14 repost of Nov 12 announcement [https://x.com/nebiustf?lang=en](https://x.com/nebiustf?lang=en)

2. **DeepSeek-V4.1-Flash-1017** - Added **Oct 23, 2026**
   - Announced as: "DeepSeek-V4.1-Flash-1017 is now available on Nebius Token Factory" on 10/23/2026 [https://x.com/nEBiUstF](https://x.com/nEBiUstF)
   - Described with DeepSeek quote: "🚀 DeepSeek-V4.1-Flash-1017 is here!" on 10/17/2026, with Nebius availability 10/23/2026 [https://x.com/nEBiUstF](https://x.com/nEBiUstF)
   - Paired announcement via LinkedIn: "Two new models are now available on Nebius Token Factory: → DeepSeek-V4.1-Flash-1017..." [https://www.linkedin.com/posts/nebius_token-factory-has-two-new-models-activity-7520871520349536259-AxtV](https://www.linkedin.com/posts/nebius_token-factory-has-two-new-models-activity-7520871520349536259-AxtV)

3. **MiniMax M3.1 Flash** - Added **Oct 23, 2026**
   - Announced as: "We’re happy to announce that MiniMax M3.1 Flash is available... on Day 0 of its open-weight release." Posted: 2026-10-23T07:27:27+00:00 [https://x.com/nebiustf/status/2113532741773881531](https://x.com/nebiustf/status/2113532741773881531)
   - Confirmed same day: "We’re happy to announce that MiniMax M3.1 Flash is available... Day 0" with 7:27 AM · Oct 23, 2026 timestamp [https://x.com/nebiustf/status/2113532741773881531/media_tags](https://x.com/nebiustf/status/2113532741773881531/media_tags)
   - Also included in LinkedIn paired announcement [https://www.linkedin.com/posts/nebius_token-factory-has-two-new-models-activity-7520871520349536259-AxtV](https://www.linkedin.com/posts/nebius_token-factory-has-two-new-models-activity-7520871520349536259-AxtV)

> Previous batch just before these was the three US open-weight models — **Reflection Beam, Muse Spark Open, and NVIDIA Nemotron 3.5 Super VL** — announced for the week of **Oct 16, 2026** [https://nebius.com/blog/posts/three-new-open-weight-models-from-us-companies-give-builders-more-choice](https://nebius.com/blog/posts/three-new-open-weight-models-from-us-companies-give-builders-more-choice) and [https://www.linkedin.com/posts/nebius_three-new-us-open-weight-models-day-0-availability-activity-7516865298566660098-U5Xb](https://www.linkedin.com/posts/nebius_three-new-us-open-weight-models-day-0-availability-activity-7516865298566660098-U5Xb), which were superseded by the Oct 23 and Nov 12 releases above.

## 3. A use case: competitor brief

The same agent handles a multi-step task. It searches with recency filters, extracts official pages in full, and writes a cited brief. Swap the company name and run it again.

In [10]:
ask(
    web_agent,
    """Write a one-page brief on Palo Alto Networks.
Cover: what it does, pricing, the two closest competitors, and news from the last 30 days.""",
)

→ tavily_search {'query': 'Palo Alto Networks what does company do products services', 'search_depth': 'advanced'}
→ tavily_search {'query': 'Palo Alto Networks pricing plans 2026', 'search_depth': 'advanced'}
→ tavily_search {'query': 'Palo Alto Networks closest competitors', 'search_depth': 'advanced'}
→ tavily_search {'query': 'Palo Alto Networks news October November 2026', 'search_depth': 'advanced', 'time_range': 'month', 'topic': 'news'}
← tavily_search [{"url": "https://costbench.com/software/cybersecurity/palo-alto-netw…", "title": "Palo Alto Networks Pricing 2026: $6.75-$18/endpoint", "score": 0.84632385, "…
← tavily_search [{"url": "https://www.polaris7.io/companies/palo-alto-networks", "title": "Palo Alto Networks - B2B SaaS Provider for Cybersecurity", "score": 0.879043, "conte…
← tavily_search [{"url": "https://tekysinfo.com/top-5-palo-alto-networks-competitors", "title": "Palo Alto Networks Competitors: What I Learned From 5 Weeks…", "score": 0.8095…
← tavily_search [{"u

One-Page Brief: Palo Alto Networks (as of Nov 23, 2026)

### 1. What It Does

Palo Alto Networks is a US-based publicly listed cybersecurity company that sells enterprise security software, cloud-delivered platforms and professional security services [https://www.polaris7.io/companies/palo-alto-networks](https://www.polaris7.io/companies/palo-alto-networks).

It is an American multinational cybersecurity company with headquarters in Santa Clara, California [https://it-dialog.com.ua/en/vendors/palo-alto-networks.html](https://it-dialog.com.ua/en/vendors/palo-alto-networks.html).

Its main product families cover network security, cloud security, secure access, security operations and AI security, while Unit 42 provides incident response, threat intelligence and managed SOC services [https://www.polaris7.io/companies/palo-alto-networks](https://www.polaris7.io/companies/palo-alto-networks).

The portfolio is grouped into three pillars and a services arm. Strata covers network security, including hardware, virtual, and cloud-delivered firewalls plus the AI-Powered Network Security Platform. Prisma covers cloud and remote-access security: Prisma SASE bundles ZTNA, Secure Web Gateway, CASB, and SD-WAN, while Prisma Cloud is the company's CNAPP [https://companyview.io/company/palo-alto-networks](https://companyview.io/company/palo-alto-networks).

The company serves over 60,000 organizations in over 150 countries, including 85 of the Fortune 100 [https://it-dialog.com.ua/en/vendors/palo-alto-networks.html](https://it-dialog.com.ua/en/vendors/palo-alto-networks.html). It operates a B2B cybersecurity platform model, developing proprietary security software and cloud services and selling them to enterprises on recurring contracts [https://www.polaris7.io/companies/palo-alto-networks](https://www.polaris7.io/companies/palo-alto-networks).

### 2. Pricing

Palo Alto Networks pricing starts at $6.75/endpoint/month across 3 plans [https://costbench.com/software/cybersecurity/palo-alto-networks](https://costbench.com/software/cybersecurity/palo-alto-networks).

Palo Alto Networks offers 3 pricing tiers: Cortex XDR Prevent, Cortex XDR Pro, Cortex XDR Pro + Managed Threat Hunting [https://costbench.com/software/cybersecurity/palo-alto-networks](https://costbench.com/software/cybersecurity/palo-alto-networks).

Paid plans include Cortex XDR Pro at $6.75/endpoint/month [https://costbench.com/software/cybersecurity/palo-alto-networks](https://costbench.com/software/cybersecurity/palo-alto-networks).

Palo Alto Networks costs $6.75 to $18 per endpoint/month [https://costbench.com/software/cybersecurity/palo-alto-networks/hidden-costs](https://costbench.com/software/cybersecurity/palo-alto-networks/hidden-costs). There is No free tier available [https://costbench.com/software/cybersecurity/palo-alto-networks](https://costbench.com/software/cybersecurity/palo-alto-networks).

At platform level, NGFW hardware from $5,000 to $200,000+, Prisma Access at $60 to $150 per user [https://www.isvcosell.com/benchmarks/vendors/palo-alto-networks-pricing](https://www.isvcosell.com/benchmarks/vendors/palo-alto-networks-pricing), and Typical discount: 10 to 55% [https://www.isvcosell.com/benchmarks/vendors/palo-alto-networks-pricing](https://www.isvcosell.com/benchmarks/vendors/palo-alto-networks-pricing), with 35 to 55% Platformization discounts [https://www.isvcosell.com/benchmarks/vendors/palo-alto-networks-pricing](https://www.isvcosell.com/benchmarks/vendors/palo-alto-networks-pricing).

Pricing depends on your chosen tier, contract length, and negotiated discounts [https://costbench.com/software/cybersecurity/palo-alto-networks/hidden-costs](https://costbench.com/software/cybersecurity/palo-alto-networks/hidden-costs) and Hardware costs separate from licensing: $10,000-$20,000 per appliance [https://costbench.com/software/cybersecurity/palo-alto-networks/calculator](https://costbench.com/software/cybersecurity/palo-alto-networks/calculator).

### 3. Two Closest Competitors

Palo Alto Networks's top 3 competitors are Cisco, Check Point, Fortinet [https://www.owler.com/company/paloaltonetworks/competitors](https://www.owler.com/company/paloaltonetworks/competitors).

1. **Cisco Systems** - Cisco is the closest NGFW competitor. Its Firepower family and SecureX platform have deep packet inspection, intrusion prevention, and SD-WAN – similar specs to the Strata line of Palo Alto [https://tekysinfo.com/top-5-palo-alto-networks-competitors](https://tekysinfo.com/top-5-palo-alto-networks-competitors). Cisco is Palo Alto Networks's #1 competitor [https://www.owler.com/company/paloaltonetworks/competitors](https://www.owler.com/company/paloaltonetworks/competitors).

2. **Fortinet (FTNT)** - Both companies offer integrated cybersecurity platforms and compete directly in network security, SASE, and security operations [https://www.hudson-labs.com/research/palo-alto-networks-competitors-panw-peers-2026](https://www.hudson-labs.com/research/palo-alto-networks-competitors-panw-peers-2026). Fortinet emphasizes its Security Fabric, secure networking hardware, and ability to integrate new features, while Palo Alto Networks focuses on platformization and consolidation of point products [https://www.hudson-labs.com/research/palo-alto-networks-competitors-panw-peers-2026](https://www.hudson-labs.com/research/palo-alto-networks-competitors-panw-peers-2026).

### 4. News – Last 30 Days (Oct 24 – Nov 23, 2026)

**Acquisition of Applied Compute for $4.5 Billion (Nov 18, 2026):** The 23-year-old Stanford dropout who sold his AI startup to Palo Alto Networks for $4.5 billion [https://www.moneycontrol.com/news/business/startup/who-is-yash-patil-the-23-year-old-stanford-dropout-who-sold-his-ai-startup-to-palo-alto-for-4-5-billion-14084804.html](https://www.moneycontrol.com/news/business/startup/who-is-yash-patil-the-23-year-old-stanford-dropout-who-sold-his-ai-startup-to-palo-alto-for-4-5-billion-14084804.html). As part of the acquisition, Yash Patil will join Palo Alto Networks as co-chief AI officer, alongside the rest of the Applied Compute team [https://www.moneycontrol.com/news/business/startup/who-is-yash-patil-the-23-year-old-stanford-dropout-who-sold-his-ai-startup-to-palo-alto-for-4-5-billion-14084804.html](https://www.moneycontrol.com/news/business/startup/who-is-yash-patil-the-23-year-old-stanford-dropout-who-sold-his-ai-startup-to-palo-alto-for-4-5-billion-14084804.html). In early November, his one-year-old AI startup Applied Compute raised $350 million at a $3.25 billion valuation after hitting a $100 million annualised revenue run rate [https://www.moneycontrol.com/news/business/startup/who-is-yash-patil-the-23-year-old-stanford-dropout-who-sold-his-ai-startup-to-palo-alto-for-4-5-billion-14084804.html](https://www.moneycontrol.com/news/business/startup/who-is-yash-patil-the-23-year-old-stanford-dropout-who-sold-his-ai-startup-to-palo-alto-for-4-5-billion-14084804.html). Palo Alto Networks has agreed to buy Applied Compute in an all-stock deal valued at US$4.5 billion [https://www.owler.com/company/paloaltonetworks/competitors](https://www.owler.com/company/paloaltonetworks/competitors).

**Critical PHP2Shell Vulnerabilities in PAN-OS (Oct 24-30, 2026):** Palo Alto Networks has confirmed that two critical PHP remote code execution vulnerabilities affect PAN-OS [https://cybersecuritynews.com/palo-alto-confirms-php-rce-vulnerabilities/](https://cybersecuritynews.com/palo-alto-confirms-php-rce-vulnerabilities/). Tracked as CVE-2026-109618 and CVE-2026-109619, the flaws have prompted planned firewall hotfixes [https://cybersecuritynews.com/palo-alto-confirms-php-rce-vulnerabilities/](https://cybersecuritynews.com/palo-alto-confirms-php-rce-vulnerabilities/). Active exploitation has been observed targeting the Palo Alto Networks management interface exposed to the internet [https://thehackernews.com/2026/10/attackers-exploit-php2shell-flaws-in.html](https://thehackernews.com/2026/10/attackers-exploit-php2shell-flaws-in.html). Palo Alto Networks says that by default the management interface is not exposed to the internet [https://www.securityweek.com/php2shell-attacks-root-palo-alto-firewalls-but-fail-against-wordpress/](https://www.securityweek.com/php2shell-attacks-root-palo-alto-firewalls-but-fail-against-wordpress/).

**Partner Event:** Savant Solutions — Coworking Day with Palo Alto Networks - Securing the AI-Driven Endpoint: Introducing Koi Agentic Endpoint Security Presented by Savant Solutions & Palo Alto Networks. Join us November 19 at Savant Solutions Downtown [https://insider.govtech.com/california/events/savant-solutions-coworking-day-with-palo-alto-networks](https://insider.govtech.com/california/events/savant-solutions-coworking-day-with-palo-alto-networks).
